# Agent Memory: Write, Read, Update, Forget

The full code of [Agent Memory: Write, Read, Update, Forget](https://residual-stream.lvtan71.workers.dev/posts/agent-memory/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/agent-memory) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/agent-memory.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.
Downloads a small sentence-embedding model (about 90 MB).

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.17.*" sentence-transformers

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "agent-memory"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `retrieval.py`

250 synthetic 50-turn conversations (5 seeds × 50) with 8 facts about the user, 3 of them changed later. For each retrieval policy (last k turns, most relevant k, and the recency + importance + relevance score with two ablations) and budget k, the share of questions whose shown memories hold the current version only, both, the outdated version only, or neither; then one worked example's three scores.

In [ ]:
!{sys.executable} retrieval.py

The post printed:

```text
                                    k  current only   both  outdated only  neither
last k turns                        1            2%     0%             0%      98%
last k turns                        3            5%     0%             0%      95%
last k turns                        5            9%     0%             0%      91%
most relevant k                     1           71%     0%            29%       0%
most relevant k                     3           62%    35%             2%       0%
most relevant k                     5           62%    36%             1%       0%
recency + importance + relevance    1           87%     0%             4%       9%
recency + importance + relevance    3           63%    36%             0%       0%
recency + importance + relevance    5           62%    38%             0%       0%
recency + relevance                 1           60%     0%             3%      37%
importance + relevance              1           71%     0%            29%       0%

worked: "What is the user's job?"
  turn 25 'Actually, things changed: I work as a data engineer.'   recency 0.45 + importance 1.00 + relevance 0.84 = 2.29
  turn 41 'Actually, things changed: My pet is a hamster.'         recency 0.80 + importance 1.00 + relevance 0.30 = 2.10
  turn  4 'I work as a teacher.'                                   recency 0.06 + importance 1.00 + relevance 1.00 = 2.06
```